<a href="https://colab.research.google.com/github/choweyyy/DS-Final-Project/blob/main/Copy_of_Final_Project_DS_58.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Health Cost EDA Project
This notebook is organized into four stages: **Load → Clean → Features → Analysis**.


## 1) Load
Load dependencies and dataset, then inspect the first rows.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
url = 'https://drive.google.com/uc?export=download&id=1SOMpQNhr5CPDdj5flrMUdwMEQVYeehaa'
df = pd.read_csv(url)
df

In [ ]:
df.head()

## 2) Clean
Run core data quality checks: schema, summary stats, missing values, duplicates, and base distributions.


In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print("=== SHAPE ===")
print(df.shape)

print("\n=== INFO ===")
print(df.info())

print("\n=== HEAD ===")
print(df.head())

# ==============================
# 4. STATISTICAL SUMMARY
# ==============================
print("\n=== DESCRIBE ===")
print(df.describe(include='all'))

# ==============================
# 5. MISSING VALUES
# ==============================
print("\n=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_percent
})

print(missing_df)

# ==============================
# 6. DUPLICATE CHECK
# ==============================
print("\n=== DUPLICATES ===")
print(df.duplicated().sum())

# ==============================
# 7. DATA TYPES CHECK
# ==============================
print("\n=== DATA TYPES ===")
print(df.dtypes)

# ==============================
# 8. CORRELATION MATRIX
# ==============================
corr = df.corr(numeric_only=True)

print("\n=== CORRELATION ===")
print(corr["annual_medical_cost"].sort_values(ascending=False))

# ==============================
# 9. DISTRIBUTION PLOTS
# ==============================
numeric_cols = [
    "annual_medical_cost",
    "monthly_premium",
    "risk_score"
]

for col in numeric_cols:
    plt.figure()
    df[col].hist()
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

# ==============================
# 10. OUTLIER CHECK (BOXPLOT)
# ==============================
for col in numeric_cols:
    plt.figure()
    plt.boxplot(df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

# ==============================
# 11. CATEGORICAL ANALYSIS
# ==============================
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    print(f"\n=== VALUE COUNTS: {col} ===")
    print(df[col].value_counts())

# ==============================
# 12. GROUP ANALYSIS (IMPORTANT)
# ==============================
# Example: high risk vs low risk
if "is_high_risk" in df.columns:
    print("\n=== COST BY RISK GROUP ===")
    print(df.groupby("is_high_risk")["annual_medical_cost"].mean())

# Example: chronic condition impact
if "chronic_count" in df.columns:
    print("\n=== COST BY CHRONIC COUNT ===")
    print(df.groupby("chronic_count")["annual_medical_cost"].mean())


## 3) Features
Create engineered fields used later in analysis (outliers, categories, encodings, disease aggregates).


In [ ]:
# Define IQR threshold for annual medical cost outliers
q1 = df["annual_medical_cost"].quantile(0.25)
q3 = df["annual_medical_cost"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr

outliers = df[df["annual_medical_cost"] > upper]

outliers[[
    "annual_medical_cost",
    "risk_score",
    "chronic_count",
    "hospitalizations_last_3yrs"
]].sort_values(by="annual_medical_cost", ascending=False).head(10)


In [ ]:
df["is_outlier"] = (df["annual_medical_cost"] > upper)

df.groupby("is_outlier")[[
    "annual_medical_cost",
    "risk_score",
    "chronic_count"
]].mean()

In [ ]:
df["outlier_type"] = "normal"

df.loc[(df["is_outlier"] == True) & (df["chronic_count"] >= 2), "outlier_type"] = "expected"

df.loc[(df["is_outlier"] == True) & (df["chronic_count"] < 2), "outlier_type"] = "unexpected"

In [ ]:
df["cost_group"] = pd.qcut(df["annual_medical_cost"], q=3, labels=["Low", "Mid", "High"])

In [ ]:
disease_cols = [
    "hypertension",
    "diabetes",
    "asthma",
    "copd",
    "cardiovascular_disease",
    "cancer_history",
    "kidney_disease",
    "liver_disease",
    "arthritis",
    "mental_health"
]

for col in disease_cols:
    print(f"\n{col}")
    print(df.groupby(col)["annual_medical_cost"].mean())

In [ ]:
df["disease_count"] = df[disease_cols].sum(axis=1)

In [ ]:
df_encoded = pd.get_dummies(df, columns=[
    "sex", "region", "urban_rural",
    "marital_status", "employment_status",
    "smoker", "plan_type", "network_tier"
], drop_first=True)

In [ ]:
education_map = {
    "No HS": 0,
    "HS": 1,
    "Some College": 2,
    "Bachelors": 3,
    "Masters": 4,
    "Doctorate": 5
}

df_encoded["education"] = df["education"].map(education_map)

In [ ]:
df_encoded["alcohol_freq"] = df["alcohol_freq"].fillna("Unknown")

In [ ]:
alcohol_map = {
    "Unknown": 0,
    "Occasional": 1,
    "Weekly": 2,
    "Daily": 3
}

df_encoded["alcohol_freq"] = df_encoded["alcohol_freq"].map(alcohol_map)


## 4) Analysis
Analyze distributions, outlier behavior, risk/cost segments, disease relationships, and encoded correlations.


In [ ]:
plt.scatter(df["risk_score"], df["annual_medical_cost"])
plt.xlabel("Risk Score")
plt.ylabel("Annual Cost")
plt.title("Outlier Detection")
plt.show()

In [ ]:
df.groupby("outlier_type")[[
    "risk_score",
    "chronic_count",
    "days_hospitalized_last_3yrs"
]].mean()

In [ ]:
df["outlier_type"].value_counts()

In [ ]:
df["outlier_type"].value_counts(normalize=True)

In [ ]:
df.groupby("outlier_type")["annual_medical_cost"].mean()

In [ ]:
df.groupby("outlier_type")[[
    "visits_last_year",
    "medication_count",
    "proc_surgery_count"
]].mean()

In [ ]:
df[df["outlier_type"]=="unexpected"].describe()

In [ ]:
sns.kdeplot(data=df, x="risk_score", hue="outlier_type", fill=True)

In [ ]:
df.groupby("outlier_type")[[
    "bmi",
    "income",
    "age"
]].mean()

In [ ]:
df.groupby(pd.cut(df["risk_score"], bins=5))["high_cost"].mean()

In [ ]:
df["annual_medical_cost"].hist(bins=50)

In [ ]:
df.groupby("cost_group")[[
    "risk_score",
    "chronic_count",
    "visits_last_year"
]].mean()

In [ ]:
cols = ["risk_score", "chronic_count", "visits_last_year"]

for col in cols:
    df.plot.scatter(x=col, y="annual_medical_cost")

In [ ]:
for col in disease_cols:
    print(f"\n{col}")
    print(df.groupby(col)["high_cost"].mean())

In [ ]:
for col in disease_cols:
    print(f"\n{col}")
    print(pd.crosstab(df[col], df["outlier_type"], normalize="index"))

In [ ]:
df.groupby("disease_count")["high_cost"].mean()

In [ ]:
df.groupby([
    pd.cut(df["risk_score"], bins=5),
    pd.cut(df["disease_count"], bins=4)
])["high_cost"].mean()

In [ ]:
pivot = df.groupby([
    pd.cut(df["risk_score"], bins=5),
    pd.cut(df["disease_count"], bins=4)
])["high_cost"].mean().unstack()
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Reds")
plt.show()

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns
print(categorical_cols)


In [ ]:
df.groupby("chronic_count")["annual_medical_cost"].mean()

In [ ]:
df.groupby("is_high_risk")["annual_medical_cost"].mean()

In [ ]:
df_encoded

In [ ]:
df_encoded.isna().sum()

In [ ]:
df_encoded.corr(numeric_only=True)["annual_medical_cost"].sort_values(ascending=False).head(10)